<h3 style="text-align:center;
    background-color:#5AA647;
    color:white;
    padding:8px;
    border-radius:6px;">
    Popularity Recommendations </h3>

In [1]:
from IPython.display import display, HTML

display(HTML("""
<div style="font-family: Arial; width: 700px; margin: auto;">

<h3 style="
    text-align:center;
    background-color:#5AA647;
    color:white;
    padding:10px;
    border-radius:6px;
">
🔥 Top Action Movies
</h3>

<table style="
    width:100%;
    border-collapse: collapse;
    margin-top:10px;
">

<tr style="background-color:#E8F5E9;">
    <th style="padding:10px; text-align:center; border-bottom:2px solid #ccc;">Movie ID</th>
    <th style="padding:10px; text-align:center; border-bottom:2px solid #ccc;">Movie Title</th>
    <th style="padding:10px; text-align:center; border-bottom:2px solid #ccc;">Avg Rating</th>
</tr>

<tr style="background-color:#d0f0c0;">
    <td style="padding:8px;">50</td>
    <td style="padding:8px;">Star Wars (1977)</td>
    <td style="padding:8px;">4.36</td>
</tr>

<tr style="background-color:#f9f9f9;">
    <td style="padding:8px;">174</td>
    <td style="padding:8px;">Raiders of the Lost Ark (1981)</td>
    <td style="padding:8px;">4.25</td>
</tr>

<tr>
    <td style="padding:8px;">511</td>
    <td style="padding:8px;">Lawrence of Arabia (1962)</td>
    <td style="padding:8px;">4.23</td>
</tr>

<tr style="background-color:#f9f9f9;">
    <td style="padding:8px;">172</td>
    <td style="padding:8px;">Empire Strikes Back, The (1980)</td>
    <td style="padding:8px;">4.20</td>
</tr>

</table>

<p style="margin-top:10px; color:#555;">
✅ Highlighted row = Most popular movie (highest rating count)
</p>

</div>
<div style="font-family: Arial; width: 700px; margin: auto;">

    <!-- Header -->
    <div style="
        text-align:center;
        background-color:#6BAE45;
        color:white;
        padding:12px;
        border-radius:8px;
        font-size:22px;
        font-weight:bold;
    ">
        🔥 Top Adventure Movies
    </div>

    <!-- Table -->
    <table style="
        width:100%;
        border-collapse: collapse;
        margin-top:15px;
    ">

        <!-- Column Headers -->
        <tr style="background-color:#EEF7ED;">
            <th style="padding:12px; text-align:center;">Movie ID</th>
            <th style="padding:12px; text-align:center;">Movie Title</th>
            <th style="padding:12px; text-align:center;">Avg Rating</th>
        </tr>

        <!-- Highlighted Top Movie -->
        <tr style="background-color:#D9F5C7;">
            <td style="padding:10px; text-align:center;">50</td>
            <td style="padding:10px;">Star Wars (1977)</td>
            <td style="padding:10px; text-align:center;">4.36</td>
        </tr>

        <!-- Alternate Rows -->
        <tr>
            <td style="padding:10px; text-align:center;">174</td>
            <td style="padding:10px;">Raiders of the Lost Ark (1981)</td>
            <td style="padding:10px; text-align:center;">4.25</td>
        </tr>

        <tr style="background-color:#F9F9F9;">
            <td style="padding:10px; text-align:center;">511</td>
            <td style="padding:10px;">Lawrence of Arabia (1962)</td>
            <td style="padding:10px; text-align:center;">4.23</td>
        </tr>

        <tr>
            <td style="padding:10px; text-align:center;">172</td>
            <td style="padding:10px;">Empire Strikes Back, The (1980)</td>
            <td style="padding:10px; text-align:center;">4.20</td>
        </tr>

    </table>

    <!-- Footer -->
    <p style="margin-top:10px; color:#555;">
        ✅ Highlighted row indicates the most popular Adventure movie
    </p>

</div>
"""))

Movie ID,Movie Title,Avg Rating
50,Star Wars (1977),4.36
174,Raiders of the Lost Ark (1981),4.25
511,Lawrence of Arabia (1962),4.23
172,"Empire Strikes Back, The (1980)",4.20
Movie ID,Movie Title,Avg Rating
50,Star Wars (1977),4.36
174,Raiders of the Lost Ark (1981),4.25
511,Lawrence of Arabia (1962),4.23
172,"Empire Strikes Back, The (1980)",4.20


#### Load the Dataset

In [2]:
import pandas as pd

# Load movies
i_cols = [
    'movie id', 'movie title', 'release date', 'video release date', 'IMDb URL',
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
    'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

items = pd.read_csv(
    'ml-100k/u.item',
    sep='|',
    names=i_cols,
    encoding='latin-1'
)

# Load ratings
r_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv(
    'ml-100k/u.data',
    sep='\t',
    names=r_cols
)

#### Popularity Based Reocommendation

In [3]:
# ✅ Dynamically extract genre columns
genre_columns = [
    col for col in items.columns
    if col not in [
        'movie id', 'movie title', 'release date',
        'video release date', 'IMDb URL', 'unknown'
    ]
]

print("Detected Genres:")
print(genre_columns)

Detected Genres:
['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [4]:
# ✅ Force genre columns to int (CRITICAL)
for col in genre_columns:
    items[col] = items[col].astype(int)

In [5]:
# ✅ Popularity metrics
movie_popularity = ratings.groupby('movie_id').agg(
    rating_count=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()


movie_popularity = movie_popularity.merge(
    items[['movie id', 'movie title'] + genre_columns],
    left_on='movie_id',
    right_on='movie id',
    how='left'
)

# ✅ DROP duplicate column immediately
movie_popularity.drop(columns=['movie id'], inplace=True)

In [6]:
print(movie_popularity.columns)

Index(['movie_id', 'rating_count', 'avg_rating', 'movie title', 'Action',
       'Adventure', 'Animation', 'Children's', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
       'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'],
      dtype='object')


In [7]:
def top_movies_by_genre(movie_popularity, genre, top_n=5):

    # ✅ This now works
    genre_movies = movie_popularity[movie_popularity[genre] == 1]

    top_movies = genre_movies.sort_values(
        by=[ 'avg_rating'],
        ascending=False
    ).head(top_n)

    return top_movies[['movie_id', 'movie title',  'avg_rating']]

In [8]:
print(movie_popularity.columns)

Index(['movie_id', 'rating_count', 'avg_rating', 'movie title', 'Action',
       'Adventure', 'Animation', 'Children's', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
       'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'],
      dtype='object')


In [9]:
def top_movies_by_genre(movie_popularity, genre, top_n=5):

    # Filter movies of this genre
    genre_movies = movie_popularity[movie_popularity[genre] == 1]

    # Rank by popularity
    top_movies = genre_movies.sort_values(
        by=['avg_rating'],
        ascending=False
    ).head(top_n)

    return top_movies[['movie_id', 'movie title','avg_rating']]

In [10]:
popular_by_genre = {}

for genre in genre_columns:
    popular_by_genre[genre] = top_movies_by_genre(
        movie_popularity, genre, top_n=5
    )

#### Style and Print the result

In [11]:
def style_popularity_table(df, title=None):

    styled = (
        df.style
        .set_caption(title)

        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("font-size", "18px"),
                    ("font-weight", "bold"),
                    ("margin-bottom", "10px"),
                    ("text-align", "left")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#f2f2f2"),
                    ("text-align", "center"),
                    ("font-weight", "bold"),
                    ("border-bottom", "2px solid #aaa"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "tbody tr:hover",
                "props": [("background-color", "#eef7ff")]
            }
        ])

        .apply(
            lambda x: [
                "background-color: #f9f9f9" if i % 2 else ""
                for i in range(len(x))
            ],
            axis=0
        )

        .highlight_max(
            subset=["avg_rating"],
            color="#d0f0c0"
        )

        .set_properties(**{
            "text-align": "left",
            "padding": "8px",
            "border": "1px solid #ddd"
        })

        .format({
            "avg_rating": "{:.3f}"
        })

        .hide(axis="index")
    )

    return styled

In [12]:
print("📊 Popular Movies for New Users (By Genre)\n")

for genre, df in popular_by_genre.items():
    # ✅ Reset index to avoid double-number confusion
    df = df.reset_index(drop=True)

    display(
        style_popularity_table(
            df,
            f"🔥 Top {genre} Movies"
        )
    )

📊 Popular Movies for New Users (By Genre)



movie_id,movie title,avg_rating
50,Star Wars (1977),4.358
127,"Godfather, The (1972)",4.283
174,Raiders of the Lost Ark (1981),4.252
313,Titanic (1997),4.246
172,"Empire Strikes Back, The (1980)",4.204


movie_id,movie title,avg_rating
1293,Star Kid (1997),5.000
50,Star Wars (1977),4.358
174,Raiders of the Lost Ark (1981),4.252
511,Lawrence of Arabia (1962),4.231
172,"Empire Strikes Back, The (1980)",4.204


movie_id,movie title,avg_rating
408,"Close Shave, A (1995)",4.491
169,"Wrong Trousers, The (1993)",4.466
114,Wallace & Gromit: The Best of Aardman Animation (1996),4.448
1367,Faust (1994),4.200
189,"Grand Day Out, A (1992)",4.106


movie_id,movie title,avg_rating
1293,Star Kid (1997),5.000
132,"Wizard of Oz, The (1939)",4.077
8,Babe (1995),3.995
500,Fly Away Home (1996),3.903
1,Toy Story (1995),3.878


...

movie_id,movie title,avg_rating
483,Casablanca (1942),4.457
50,Star Wars (1977),4.358
313,Titanic (1997),4.246
172,"Empire Strikes Back, The (1980)",4.204
966,"Affair to Remember, An (1957)",4.192


movie_id,movie title,avg_rating
1293,Star Kid (1997),5.000
50,Star Wars (1977),4.358
474,Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963),4.253
172,"Empire Strikes Back, The (1980)",4.204
89,Blade Runner (1982),4.138


movie_id,movie title,avg_rating
408,"Close Shave, A (1995)",4.491
603,Rear Window (1954),4.388
12,"Usual Suspects, The (1995)",4.386
513,"Third Man, The (1949)",4.333
963,Some Folks Call It a Sling Blade (1993),4.293


movie_id,movie title,avg_rating
318,Schindler's List (1993),4.466
483,Casablanca (1942),4.457
50,Star Wars (1977),4.358
474,Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963),4.253
511,Lawrence of Arabia (1962),4.231


movie_id,movie title,avg_rating
661,High Noon (1952),4.102
589,"Wild Bunch, The (1969)",4.023
435,Butch Cassidy and the Sundance Kid (1969),3.949
510,"Magnificent Seven, The (1954)",3.942
646,Once Upon a Time in the West (1969),3.868
